# Stage 5 Flashover Binary Ablation

Targeted normal-vs-flashover diagnostic. Thresholds/classifier settings are selected by train-CV inside the baseline script.

In [ ]:
from pathlib import Path
import subprocess, shutil, pandas as pd
from IPython.display import display
REPO_URL='https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git'
REPO_BRANCH='next_research_vlm_benefit'
REPO_DIR=Path('/kaggle/working/vlm-for-insulator-defect-detection')
DATA_ROOTS=[Path('/kaggle/input/datasets/kostyaryazanov/idid-coco-v3'), Path('/kaggle/input/idid-coco-v3')]
TRAIN_REL=Path('stage3_regrouped_v2/train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl')
VAL_REL=Path('stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl')
ARCHIVE=Path('/kaggle/working/stage5_flashover_binary_ablation.tar.gz')

def sh(args,cwd=None,check=True):
    print('$',' '.join(str(a) for a in args))
    p=subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout: print(p.stdout)
    if p.stderr: print(p.stderr)
    if check and p.returncode!=0: raise RuntimeError(f'Command failed {p.returncode}')
    return p

def data_root():
    for r in DATA_ROOTS:
        if (r/TRAIN_REL).exists() and (r/VAL_REL).exists(): return r
    raise FileNotFoundError('stage3_regrouped_v2 train/val JSONL not found')


In [ ]:
if sh(['nvidia-smi'], check=False).returncode != 0:
    raise RuntimeError('GPU is required')
DATA_ROOT=data_root()
print('DATA_ROOT:', DATA_ROOT)
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
sh(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,REPO_DIR])
sh(['python','-m','pip','install','-q','transformers==4.57.1','accelerate','scikit-learn','pandas','numpy','pillow','timm','tabulate'], cwd=REPO_DIR)


In [ ]:
out_dir='reports/next_research/accuracy_ablation/flashover_binary'
sh([
    'python','scripts/run_flashover_binary_ablation.py',
    '--train-jsonl', DATA_ROOT/TRAIN_REL,
    '--val-jsonl', DATA_ROOT/VAL_REL,
    '--dataset-root', DATA_ROOT/'stage3_regrouped_v2',
    '--out-dir', out_dir,
    '--models', 'dinov2_binary=facebook/dinov2-base:16,clip_b32_binary=openai/clip-vit-base-patch32:32',
    '--classifiers', 'logreg,svm',
    '--continue-on-error',
], cwd=REPO_DIR)


In [ ]:
lb=REPO_DIR/'reports/next_research/accuracy_ablation/flashover_binary/leaderboard_non_vlm.csv'
if lb.exists():
    df=pd.read_csv(lb); display(df)
summary=REPO_DIR/'reports/next_research/accuracy_ablation/flashover_binary/summary.md'
if summary.exists(): print(summary.read_text(encoding='utf-8')[:8000])
if ARCHIVE.exists(): ARCHIVE.unlink()
sh(['tar','-czf',ARCHIVE,'-C',REPO_DIR,'reports/next_research/accuracy_ablation/flashover_binary'])
print('Archive:', ARCHIVE)
